In [1]:

#! /usr/bin/env python3
import numpy as np
import sys, os
import h5py


os.chdir("/data/jihenghu/juno-mwr-deconv-research/21.polarmean_redo/spectra/adiabat_fit")

sys.path.append("build/python")
sys.path.append(".")
from canoe import def_species, load_configure, index_map
from canoe.snap import def_thermo
from canoe.athena import Mesh, ParameterInput, Outputs, MeshBlock
# from canoe.harp import radiation_band, radiation

nx2 = 5  # Shall not be less than n_walkers, can be a little greater for safety.
global pin
pin = ParameterInput()
pin.load_from_file("juno_mwr.inp")

vapors = pin.get_string("species", "vapor").split(", ")
clouds = pin.get_string("species", "cloud").split(", ")
tracers = pin.get_string("species", "tracer").split(", ")

def_species(vapors=vapors, clouds=clouds, tracers=tracers)
def_thermo(pin)

config = load_configure("juno_mwr.yaml")

pin.set_boolean("job", "verbose", False)
pin.set_string("mesh", "nx2", f"{nx2}")


# global n_angle

# n_angle=49
# mu = np.degrees(np.arccos(np.linspace(1, 0.6, n_angle)))
# angles = ' '.join([f"({x},)" for x in mu])
# pin.set_string("radiation","outdir",angles)

print(pin.get_string("radiation","outdir"))
mesh = Mesh(pin)
mesh.initialize(pin)

global mb, rad, nb
mb = mesh.meshblock(0)
rad = mb.get_rad()
nb = rad.get_num_bands()

# global iNH3,iH2O
# pindex = index_map.get_instance()
# iNH3 = pindex.get_vapor_id("NH3")
# iH2O = pindex.get_vapor_id("H2O")
# print(iNH3)


qNH3=362.71
stdNH3=10.5

qH2O=2500
T1bar=177.54
stdT1bar=1.35
RHmax=1
adlnNH3dlnP=0.
pmax=0.
method="dry"
jindex=0

# do 1000 monte carlo simulation varying qNH3 and T1bar according to their standard deviations
n_sim = 1000
tbs_samples = np.zeros((n_sim, nb, 4))

for i in range(n_sim):
    print(i)
    qNH3_i = np.random.normal(qNH3, stdNH3)
    T1bar_i = np.random.normal(T1bar, stdT1bar)
    mb.construct_atmosphere(pin, qNH3_i, T1bar_i, RHmax, jindex, method, qH2O, 200)
    rad.cal_radiance(mb, mb.k_st, mb.j_st + jindex)
    for ib in range(nb):
        toa = rad.get_band(ib).get_toa()[0]
        tbs_samples[i, ib, :] = toa

# calculate the limb darkening
LDs = np.zeros((n_sim, nb))
ndtb = tbs_samples[:,:, 0]

for i in range(n_sim):
    for ib in range(nb):
        LDs[i, ib]  = (tbs_samples[i, ib, 0]-tbs_samples[i, ib, 3]) / tbs_samples[i, ib, 0] * 100.


# Save the samples to a file
output_file = "../best_fit_monte_carlo_1000.h5"
with h5py.File(output_file, 'w') as f:
    f.create_dataset('naTBs', data=ndtb)
    f.create_dataset('LDs', data=LDs)



(0,) (15,) (30,) (45,)Log, "2025-09-20 15:21:14",        canoe, 1., "Installing monitor canoe"

0
Log, "2025-09-20 15:21:14",        canoe, 1.1., "Initialize IndexMap"
Log, "2025-09-20 15:21:14",         snap, 3., "Installing monitor snap"
Log, "2025-09-20 15:21:14",         snap, 3.1., "Initialize Thermodynamics"
Log, "2025-09-20 15:21:14",         snap, 3.1.1., "Enrolling vapor functions"
Log, "2025-09-20 15:21:14",         snap, 3.1.1.1., "Enrolling H2O vapor pressures"
Log, "2025-09-20 15:21:14",         snap, 3.1.2.1., "Enrolling NH3 vapor pressures"
Log, "2025-09-20 15:21:14",         snap, 4.1., "Initialize Decomposition"
Log, "2025-09-20 15:21:14",         snap, 5.1., "Initialize ImplicitSolver"
Log, "2025-09-20 15:21:14", microphysics, 7., "Installing monitor microphysics"
Log, "2025-09-20 15:21:14", microphysics, 7.1., "Initialize Microphysics"
Log, "2025-09-20 15:21:14",         harp, 9., "Installing monitor harp"
Log, "2025-09-20 15:21:14",         harp, 9.1., "Initialize R

In [1]:

#! /usr/bin/env python3
import numpy as np
import sys, os
import h5py


os.chdir("/data/jihenghu/juno-mwr-deconv-research/21.polarmean_redo/spectra/adiabat_fit")

sys.path.append("build/python")
sys.path.append(".")
from canoe import def_species, load_configure, index_map
from canoe.snap import def_thermo
from canoe.athena import Mesh, ParameterInput, Outputs, MeshBlock
# from canoe.harp import radiation_band, radiation

nx2 = 5  # Shall not be less than n_walkers, can be a little greater for safety.
global pin
pin = ParameterInput()
pin.load_from_file("juno_mwr.inp")

vapors = pin.get_string("species", "vapor").split(", ")
clouds = pin.get_string("species", "cloud").split(", ")
tracers = pin.get_string("species", "tracer").split(", ")

def_species(vapors=vapors, clouds=clouds, tracers=tracers)
def_thermo(pin)

config = load_configure("juno_mwr.yaml")

pin.set_boolean("job", "verbose", False)
pin.set_string("mesh", "nx2", f"{nx2}")


# global n_angle

# n_angle=49
# mu = np.degrees(np.arccos(np.linspace(1, 0.6, n_angle)))
# angles = ' '.join([f"({x},)" for x in mu])
# pin.set_string("radiation","outdir",angles)

print(pin.get_string("radiation","outdir"))
mesh = Mesh(pin)
mesh.initialize(pin)

global mb, rad, nb
mb = mesh.meshblock(0)
rad = mb.get_rad()
nb = rad.get_num_bands()

# global iNH3,iH2O
# pindex = index_map.get_instance()
# iNH3 = pindex.get_vapor_id("NH3")
# iH2O = pindex.get_vapor_id("H2O")
# print(iNH3)


qNH3=362.71
stdNH3=10.5

qH2O=2500
T1bar=176.2 # a trial fit
stdT1bar=1.35
RHmax=1
adlnNH3dlnP=0.
pmax=0.
method="dry"
jindex=0

# do 1000 monte carlo simulation varying qNH3 and T1bar according to their standard deviations
n_sim = 1000
tbs_samples = np.zeros((n_sim, nb, 4))

for i in range(n_sim):
    print(i)
    qNH3_i = np.random.normal(qNH3, stdNH3)
    T1bar_i = np.random.normal(T1bar, stdT1bar)
    mb.construct_atmosphere(pin, qNH3_i, T1bar_i, RHmax, jindex, method, qH2O, 200)
    rad.cal_radiance(mb, mb.k_st, mb.j_st + jindex)
    for ib in range(nb):
        toa = rad.get_band(ib).get_toa()[0]
        tbs_samples[i, ib, :] = toa

# calculate the limb darkening
LDs = np.zeros((n_sim, nb))
ndtb = tbs_samples[:,:, 0]

for i in range(n_sim):
    for ib in range(nb):
        LDs[i, ib]  = (tbs_samples[i, ib, 0]-tbs_samples[i, ib, 3]) / tbs_samples[i, ib, 0] * 100.


# Save the samples to a file
output_file = "../trial_fit_monte_carlo_1000.h5"
with h5py.File(output_file, 'w') as f:
    f.create_dataset('naTBs', data=ndtb)
    f.create_dataset('LDs', data=LDs)



(0,) (15,) (30,) (45,)Log, "2025-10-06 15:47:43",        canoe, 1., "Installing monitor canoe"

Log, "2025-10-06 15:47:43",        canoe, 1.1., "Initialize IndexMap"
Log, "2025-10-06 15:47:43",         snap, 3., "Installing monitor snap"
Log, "2025-10-06 15:47:43",         snap, 3.1., "Initialize Thermodynamics"
Log, "2025-10-06 15:47:43",         snap, 3.1.1., "Enrolling vapor functions"
Log, "2025-10-06 15:47:43",         snap, 3.1.1.1., "Enrolling H2O vapor pressures"
Log, "2025-10-06 15:47:43",         snap, 3.1.2.1., "Enrolling NH3 vapor pressures"
Log, "2025-10-06 15:47:43",         snap, 4.1., "Initialize Decomposition"
Log, "2025-10-06 15:47:43",         snap, 5.1., "Initialize ImplicitSolver"
Log, "2025-10-06 15:47:43", microphysics, 7., "Installing monitor microphysics"
Log, "2025-10-06 15:47:43", microphysics, 7.1., "Initialize Microphysics"
Log, "2025-10-06 15:47:43",         harp, 9., "Installing monitor harp"
Log, "2025-10-06 15:47:43",         harp, 9.1., "Initialize Rad

In [1]:

## gernerate the EZtb


#! /usr/bin/env python3
import numpy as np
import emcee
import sys, os
import matplotlib.pyplot as plt
import h5py
from scipy.stats import norm
from tqdm import tqdm
import multiprocessing
from multiprocessing import Pool, current_process, shared_memory
import threading
import queue
import time

os.chdir("/data/jihenghu/juno-mwr-deconv-research/21.polarmean_redo/spectra/adiabat_fit")

sys.path.append("build/python")
sys.path.append(".")
from canoe import def_species, load_configure, index_map
from canoe.snap import def_thermo
from canoe.athena import Mesh, ParameterInput, Outputs, MeshBlock
# from canoe.harp import radiation_band, radiation

nx2 = 5  # Shall not be less than n_walkers, can be a little greater for safety.
global pin
pin = ParameterInput()
pin.load_from_file("juno_mwr.inp")

vapors = pin.get_string("species", "vapor").split(", ")
clouds = pin.get_string("species", "cloud").split(", ")
tracers = pin.get_string("species", "tracer").split(", ")

def_species(vapors=vapors, clouds=clouds, tracers=tracers)
def_thermo(pin)

config = load_configure("juno_mwr.yaml")

pin.set_boolean("job", "verbose", False)
pin.set_string("mesh", "nx2", f"{nx2}")


# global n_angle

# n_angle=49
# mu = np.degrees(np.arccos(np.linspace(1, 0.6, n_angle)))
# angles = ' '.join([f"({x},)" for x in mu])
# pin.set_string("radiation","outdir",angles)

print(pin.get_string("radiation","outdir"))
mesh = Mesh(pin)
mesh.initialize(pin)

global mb, rad, nb
mb = mesh.meshblock(0)
rad = mb.get_rad()
nb = rad.get_num_bands()

# global iNH3,iH2O
# pindex = index_map.get_instance()
# iNH3 = pindex.get_vapor_id("NH3")
# iH2O = pindex.get_vapor_id("H2O")
# print(iNH3)


qNH3=351
stdNH3=22
# qNH3=360
qH2O=2500
T1bar=169
stdH2O=2200
RHmax=1
adlnNH3dlnP=0.
pmax=0.
method="dry"
jindex=0


# do 1000 monte carlo simulation varying qNH3 and T1bar according to their standard deviations
n_sim = 1000
tbs_samples = np.zeros((n_sim, nb, 4))

for i in tqdm(range(n_sim)):
    # print(i)
    qNH3_i = np.random.normal(qNH3, stdNH3)
    qH2O_i = np.random.normal(qH2O, stdH2O)
    mb.construct_atmosphere(pin, qNH3_i, T1bar, RHmax, jindex, method, qH2O_i, 200)
    rad.cal_radiance(mb, mb.k_st, mb.j_st + jindex)
    for ib in range(nb):
        toa = rad.get_band(ib).get_toa()[0]
        tbs_samples[i, ib, :] = toa

# calculate the limb darkening
LDs = np.zeros((n_sim, nb))
ndtb = tbs_samples[:,:, 0]

for i in range(n_sim):
    for ib in range(nb):
        LDs[i, ib]  = (tbs_samples[i, ib, 0]-tbs_samples[i, ib, 3]) / tbs_samples[i, ib, 0] * 100.


# Save the samples to a file
output_file = "../EZ_monte_carlo_1000.h5"
with h5py.File(output_file, 'w') as f:
    f.create_dataset('naTBs', data=ndtb)
    f.create_dataset('LDs', data=LDs)



(0,) (15,) (30,) (45,)
Log, "2025-09-20 15:34:32",        canoe, 1., "Installing monitor canoe"
Log, "2025-09-20 15:34:32",        canoe, 1.1., "Initialize IndexMap"
Log, "2025-09-20 15:34:32",         snap, 3., "Installing monitor snap"
Log, "2025-09-20 15:34:32",         snap, 3.1., "Initialize Thermodynamics"
Log, "2025-09-20 15:34:32",         snap, 3.1.1., "Enrolling vapor functions"
Log, "2025-09-20 15:34:32",         snap, 3.1.1.1., "Enrolling H2O vapor pressures"
Log, "2025-09-20 15:34:32",         snap, 3.1.2.1., "Enrolling NH3 vapor pressures"
Log, "2025-09-20 15:34:32",         snap, 4.1., "Initialize Decomposition"
Log, "2025-09-20 15:34:32",         snap, 5.1., "Initialize ImplicitSolver"
Log, "2025-09-20 15:34:32", microphysics, 7., "Installing monitor microphysics"
Log, "2025-09-20 15:34:32", microphysics, 7.1., "Initialize Microphysics"
Log, "2025-09-20 15:34:32",         harp, 9., "Installing monitor harp"
Log, "2025-09-20 15:34:32",         harp, 9.1., "Initialize Rad

  0%|          | 0/1000 [00:00<?, ?it/s]

Log, "2025-09-20 15:34:34", pycanoe_construct_atmosphere, 18., "Installing monitor pycanoe_construct_atmosphere"


100%|██████████| 1000/1000 [10:26<00:00,  1.60it/s]
